# Usar y auditar un modelo preentrenado

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Qué podemos hacer con un modelo de sentimiento ya entrenado y qué debemos examinar antes de confiar en sus resultados?

## Objetivo

Aplicar un modelo de sentimiento en español que no entrenamos nosotros, registrar qué sabemos (y qué no) sobre él, y auditar sus respuestas con casos que lo ponen a prueba.

## Al terminar este cuaderno vas a poder:

1. Distinguir entrenamiento de inferencia, y explicar por qué acá solo hacemos inferencia.
2. Registrar qué modelo se usó, con qué versión y con qué limitaciones documentadas.
3. Leer etiquetas y probabilidades sin tomarlas como certezas.
4. Mostrar con evidencia cómo el preprocesamiento y el contexto cambian lo que el modelo responde.
5. Formular una recomendación de uso delimitada.

## Alcance

No entrenaremos un transformer. Aplicaremos un modelo existente y examinaremos su comportamiento.

## ◈ Marco contextual

**Analogía.** Imaginá que contratás a una persona que leyó millones de mensajes y aprendió a decir si suenan favorables, desfavorables o neutros. Vos no sabés qué leyó, ni cómo decide, ni quién eligió esos mensajes. Lo único que podés hacer es hacerle preguntas, mirar sus respuestas y fijarte cuánta seguridad muestra. Este cuaderno es esa entrevista: no la contratamos a ciegas, la **auditamos**.

**Bajada tecnológica.** Hoy casi nadie entrena desde cero un modelo de lenguaje: se descargan modelos *preentrenados* de repositorios como el Hub de Hugging Face y se aplican a datos propios. Se usan en moderación de comentarios, análisis de reseñas y atención al cliente. La biblioteca `pysentimiento` empaqueta modelos de este tipo para español. El nombre del modelo que carga (lo vamos a ver en pantalla) indica que se apoya en *RoBERTuito*, un modelo de lenguaje entrenado con mensajes de Twitter en español; confirmalo en su ficha, como pide la sección 2.

## ◈ Microglosario

| Término | Qué significa en este cuaderno |
|---|---|
| **Entrenamiento** | Ajustar un modelo con ejemplos. Es lo que hicimos en el cuaderno 02. Acá no lo hacemos. |
| **Inferencia** | Usar un modelo ya entrenado para obtener una respuesta sobre un texto nuevo. |
| **Modelo preentrenado** | Modelo que alguien más entrenó con sus datos y que descargamos tal cual. |
| **Probabilidad del modelo** | Cómo reparte el modelo su 100 % entre las clases. No es la probabilidad de que sea correcto. |
| **Auditoría** | Revisión sistemática de qué hace el modelo con casos elegidos para ponerlo a prueba. |
| **Preprocesamiento** | Cambios que le hacemos al texto antes de dárselo al modelo (por ejemplo, borrar palabras). |

## 1. Configuración del entorno

**Operación.** Importar las herramientas, registrar versiones y cargar el corpus de siempre.

**Por qué importa.** Este cuaderno depende de `torch` y `transformers` además de pandas. Si las versiones cambian, las probabilidades pueden cambiar levemente. Por eso se registran.

In [ ]:
# sys e importlib.metadata: para registrar qué versiones estamos usando.
import sys
import importlib.metadata

# re: expresiones regulares, para el experimento de limpieza de texto.
import re

# pandas: tablas. display: las muestra completas en el cuaderno.
import pandas as pd
from IPython.display import display

# utils.py (en esta carpeta): ayudas que dibujan los gráficos. No calculan nada.
from utils import ETIQUETAS, dibujar_matriz_de_confusión, dibujar_probabilidades

try:
    from pysentimiento import create_analyzer
except ImportError:
    print("Falta pysentimiento. Instalalo con: uv pip install -r requirements.txt")
    raise

pd.set_option("display.max_colwidth", 140)

print("Python:", sys.version.split()[0])
for paquete in ["pandas", "matplotlib", "torch", "transformers", "pysentimiento"]:
    print(f"{paquete}: {importlib.metadata.version(paquete)}")

**Datos.** Cargamos el corpus y los desafíos de los cuadernos anteriores.

In [ ]:
corpus = pd.read_csv("datos/corpus_sentimiento.csv")
desafíos = pd.read_csv("datos/desafios.csv")

prueba = corpus.loc[corpus["partición"] == "prueba"].copy()

assert corpus["etiqueta_referencia"].isin(ETIQUETAS).all(), "Las etiquetas solo pueden ser NEG, NEU o POS."
print(f"Textos de prueba: {len(prueba)}")
print(f"Desafíos: {len(desafíos)}")

**Leé el resultado.** Usamos los 9 textos de prueba y los 8 desafíos de los cuadernos anteriores (ficha de datos en el cuaderno 01). Los textos de entrenamiento no se usan acá: este modelo ya fue entrenado con otros datos.

**Recordatorio.** La prueba es deliberadamente fácil. Los desafíos son casos pensados para tensionar al modelo, sin etiqueta de referencia automática.

## 2. Cargar el modelo y registrar qué usamos

La primera vez que ejecutes la celda siguiente, el programa descarga el modelo desde Internet. Es un archivo grande: por eso **se descarga antes de la clase** (ver `ANTES_DE_LA_CLASE.md`). Es normal que aparezcan barras de progreso y algunos avisos.

Si aparece un error de dependencias, registrá el mensaje completo. Si el entorno solicita reiniciar el kernel, reinicialo y ejecutá nuevamente las celdas en orden. **No interpretes una falla de instalación como un error de clasificación.**

In [ ]:
import time

inicio = time.perf_counter()

# create_analyzer descarga (la primera vez) y carga el modelo de sentimiento para español.
analizador = create_analyzer(task="sentiment", lang="es")

duración = time.perf_counter() - inicio
configuración = analizador.model.config

print("Tiempo de carga en segundos:", round(duración, 1))
print("Identificador del modelo:", getattr(configuración, "_name_or_path", "No informado"))
print("Correspondencia de etiquetas:", getattr(configuración, "id2label", "No informada"))

assert set(configuración.id2label.values()) == set(ETIQUETAS), "El modelo debería responder NEG, NEU o POS."

**Leé el resultado.** El identificador es el nombre del modelo en el Hub, y la correspondencia de etiquetas confirma que responde `NEG`, `NEU` y `POS`, las mismas clases de nuestro protocolo. Eso no significa que el criterio de *neutro* sea el nuestro.

### No elegimos solo una biblioteca

La biblioteca organiza el uso del modelo, pero las predicciones dependen del modelo y de su entrenamiento. Completá esta ficha consultando la [documentación del proyecto](https://github.com/pysentimiento/pysentimiento) y la ficha del modelo que efectivamente cargó tu instalación:

| Dato | Lo que encontré |
|---|---|
| Identificador del modelo | |
| Tarea | |
| Clases de salida | |
| Dominio de sus datos de entrenamiento | |
| Limitaciones documentadas | |
| Fuente consultada | |
| Fecha de consulta | |

Si no encontrás un dato, escribí "no encontrado": no lo completes por suposición.

## 3. Una primera predicción

**Operación.** Aplicar el modelo a una sola oración y mirar qué devuelve.

La respuesta de `predict` es un objeto con dos partes: `.output` (la etiqueta elegida) y `.probas` (cómo reparte el modelo su 100 % entre las tres clases).

In [ ]:
texto = "La atención fue excelente."
resultado = analizador.predict(texto)

print("Texto:", texto)
print("Etiqueta:", resultado.output)

tabla_de_una_predicción = pd.DataFrame([{
    "texto": texto,
    "prob_NEG": resultado.probas["NEG"],
    "prob_NEU": resultado.probas["NEU"],
    "prob_POS": resultado.probas["POS"],
}])
dibujar_probabilidades(tabla=tabla_de_una_predicción, columna_texto="texto")

**Leé el gráfico.** La barra reparte el 100 % entre las tres clases. Casi todo va a `POS`.

Las probabilidades expresan la distribución de salida del modelo. No demuestran que la predicción sea correcta ni que describa la emoción de quien escribió el texto.

## 4. Evaluar con la partición de prueba

**Operación.** Escribimos una función que aplica el modelo a cada texto de una tabla y agrega cuatro columnas: la etiqueta y las tres probabilidades. La vamos a reutilizar en todo el cuaderno.

### ✎ Antes de ejecutar

Elegí tres textos de prueba. Para cada uno, anotá:

- Tu etiqueta según el protocolo.
- La expresión que justifica esa etiqueta.
- La etiqueta que esperás del modelo.
- Una razón por la que podría equivocarse.

**Mis anotaciones**

| Texto | Mi etiqueta | Expresión que la justifica | Qué espero del modelo | Por qué podría fallar |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |

In [ ]:
def analizar_textos(tabla):
    """Aplica el modelo a cada texto de una tabla.

    Recibe una tabla con una columna 'texto'. Devuelve la misma tabla con
    cuatro columnas nuevas: predicción, prob_NEG, prob_NEU y prob_POS.
    """
    filas = []
    for texto in tabla["texto"]:
        resultado = analizador.predict(texto)
        filas.append({
            "predicción": resultado.output,
            "prob_NEG": resultado.probas["NEG"],
            "prob_NEU": resultado.probas["NEU"],
            "prob_POS": resultado.probas["POS"],
        })
    resultados_del_modelo = pd.DataFrame(filas, index=tabla.index)
    return pd.concat([tabla, resultados_del_modelo], axis=1)

**Aplicación.** Usamos la función sobre los 9 textos de prueba.

In [ ]:
resultados = analizar_textos(prueba)

resultados["acierto"] = resultados["etiqueta_referencia"] == resultados["predicción"]
cantidad_de_aciertos = resultados["acierto"].sum()
print(f"Aciertos: {cantidad_de_aciertos} de {len(resultados)}")

resultados["rótulo"] = resultados["id"] + " (ref. " + resultados["etiqueta_referencia"] + "): " + resultados["texto"]
dibujar_probabilidades(
    tabla=resultados, columna_texto="rótulo",
    título="Prueba: cómo reparte el modelo su 100 % en cada texto",
)

**Los mismos resultados, en una matriz.** Las filas son la referencia y las columnas, la respuesta del modelo.

In [ ]:
dibujar_matriz_de_confusión(
    referencia=resultados["etiqueta_referencia"], predicción=resultados["predicción"],
    título="Prueba: modelo preentrenado",
)

desacuerdos = resultados.loc[resultados["etiqueta_referencia"] != resultados["predicción"]]
print("Cantidad de desacuerdos con la referencia:", len(desacuerdos))

**Leé el gráfico y la matriz.**

- **Observación:** los 9 textos coinciden con la referencia, y en todos la clase elegida concentra entre el 81 % y el 97 % de la probabilidad.
- **Interpretación posible:** son oraciones cortas y con vocabulario muy explícito (`desagradable`, `confuso`, `amable`), un terreno fácil para cualquier método. Los `NEU` son los que dejan más margen a otras clases (81 % a 91 %).
- **Límite:** igual que en los cuadernos 01 y 02, 9 de 9 en una prueba fácil no dice nada sobre textos reales. Lo interesante no es cuántos acierta, sino **dónde y por qué** deja de acertar.

## 5. Los desafíos: auditoría

**Pregunta.** ¿Qué hace el modelo con los casos que los métodos anteriores resolvieron mal?

**Operación.** Aplicamos el modelo a los 8 desafíos y los ordenamos de menor a mayor seguridad (la probabilidad máxima de cada texto).

In [ ]:
resultados_desafíos = analizar_textos(desafíos)

columnas_de_probabilidad = ["prob_NEG", "prob_NEU", "prob_POS"]
resultados_desafíos["probabilidad_máxima"] = resultados_desafíos[columnas_de_probabilidad].max(axis=1)
resultados_desafíos = resultados_desafíos.sort_values(by="probabilidad_máxima")

resultados_desafíos["rótulo"] = resultados_desafíos["id"] + " → " + resultados_desafíos["predicción"] + ": " + resultados_desafíos["texto"]
dibujar_probabilidades(
    tabla=resultados_desafíos, columna_texto="rótulo",
    título="Desafíos, de menor a mayor seguridad del modelo",
)

**Leé el gráfico.** Cada barra dice qué etiqueta eligió el modelo y con cuánta seguridad.

| Desafío | Respuesta | Qué se observa |
|---|---|---|
| D01 "no fue buena" | `NEG` (76 %) | A diferencia de los cuadernos 01 y 02 (el 02 daba `POS`), el modelo registra la negación. |
| D02 "no fue mala" | `NEU` (74 %) | Ni positivo ni negativo: una lectura matizada, cercana a la que discutimos en el 01. |
| D03 ironía | `NEG` (54 %) | La menor seguridad de la lista. Acierta la etiqueta, pero no sabemos si por detectar la ironía o por reaccionar a "se borró todo". |
| D04 dos objetos | `NEG` (72 %) | Una sola etiqueta para dos valoraciones opuestas: se pierde cuál va a quién. |
| D05 voces | `NEG` (92 %) | Alta seguridad, pero una lectura humana puede ver que quien escribe dice lo contrario ("a mí me resolvieron el problema"). |
| D06 evento desfavorable | `NEG` (69 %) | Lee como negativo un aviso sin palabras valorativas. Según el protocolo, ¿es una valoración o información? |
| D07 coloquial | `POS` (95 %) | Alta seguridad en un uso de `buena` que admite otras lecturas. |
| D08 mixta | `NEG` (74 %) | La parte positiva se pierde en la etiqueta global. |

**Límite.** Una probabilidad alta no garantiza una lectura correcta: D05 y D07 son de las más altas y ambas admiten lecturas humanas distintas. Tampoco hay forma de saber, mirando solo la salida, *por qué* el modelo decidió así.

### ✎ Auditoría

Elegí **tres desafíos**, e incluí al menos uno con probabilidad máxima alta. Para cada uno:

- ¿Qué o quién recibe la valoración?
- ¿Hay más de una voz o más de un objeto?
- ¿La etiqueta global conserva lo importante?
- ¿Qué expresión sostiene tu interpretación?
- ¿El resultado parece un error, una lectura plausible o un problema de la tarea?

No cambies la etiqueta humana únicamente porque el modelo respondió otra cosa. Si modificás tu lectura, explicá qué evidencia textual te llevó a hacerlo.

**Análisis de los tres casos:**

## 6. Efecto del preprocesamiento

**Hipótesis.** "Eliminar negaciones y conectores puede borrar diferencias relevantes para clasificar valoraciones".

**Operación.** La función `limpieza_agresiva` hace una limpieza típica de otros enfoques: pasa a minúsculas, saca los signos y borra palabras muy frecuentes, incluidas `no`, `pero` y `y`. Es **deliberadamente problemática**: no es una recomendación de preprocesamiento. Comparamos las respuestas del modelo con el texto original y con el texto limpio.

In [ ]:
PALABRAS_A_ELIMINAR = {
    "no", "nunca", "jamás", "el", "la", "los", "las",
    "un", "una", "de", "del", "que", "y", "pero"
}


def limpieza_agresiva(texto):
    """Deja solo las palabras que no están en PALABRAS_A_ELIMINAR, en minúsculas y sin signos.

    Recibe un texto. Devuelve el texto limpio.
    """
    palabras = re.findall(pattern=r"\w+", string=texto.lower())
    palabras_conservadas = []
    for palabra in palabras:
        if palabra not in PALABRAS_A_ELIMINAR:
            palabras_conservadas.append(palabra)
    return " ".join(palabras_conservadas)

**Aplicación.** Limpiamos cada texto y le pedimos al modelo una respuesta para el original y para el limpio.

In [ ]:
textos_originales = [
    "La atención fue buena.",
    "La atención no fue buena.",
    "La plataforma funciona bien, pero la atención es pésima.",
    "Buenísimo, se borró todo otra vez.",
]

textos_limpios = []
for texto_original in textos_originales:
    textos_limpios.append(limpieza_agresiva(texto_original))

resultados_originales = analizar_textos(pd.DataFrame({"texto": textos_originales}))
resultados_limpios = analizar_textos(pd.DataFrame({"texto": textos_limpios}))

comparación = pd.DataFrame({
    "original": textos_originales,
    "limpio": textos_limpios,
    "etiqueta original": resultados_originales["predicción"],
    "etiqueta limpia": resultados_limpios["predicción"],
})
display(comparación)

**En gráficos.** Cómo reparte el modelo su 100 % antes y después de la limpieza.

In [ ]:
dibujar_probabilidades(tabla=resultados_originales, columna_texto="texto", título="Texto original")
dibujar_probabilidades(tabla=resultados_limpios, columna_texto="texto", título="Texto después de la limpieza agresiva")

**Leé la tabla y los gráficos.**

- **Observación:** "La atención fue buena" y "La atención no fue buena" quedan **idénticas** después de la limpieza (`atención fue buena`), y el modelo pasa de `NEG` a `POS` en la segunda. En "Buenísimo, se borró todo otra vez", la etiqueta no cambia, pero la probabilidad de `NEG` sube del 54 % al 78 %.
- **Interpretación posible:** al borrar `no` se destruyó la diferencia entre dos oraciones de sentido opuesto.
- **Límite:** que la etiqueta no cambie (caso 3 y caso 4) no demuestra que la información borrada fuera irrelevante: cambiaron las probabilidades, y con otro modelo o con otros textos podría cambiar la etiqueta.

### ✎ Para pensar

1. ¿Qué información se eliminó? ¿Dos textos diferentes quedaron iguales?
2. Distinguí el efecto sobre el texto del efecto observado sobre el modelo.

Respuesta:

## 7. Efecto del contexto

**Pregunta.** ¿La misma frase recibe la misma valoración si la ponemos dentro de un texto más largo?

**Operación.** Para cada una de tres frases, comparamos la respuesta del modelo con la frase sola y con una versión que agrega contexto.

In [ ]:
pares_de_contexto = pd.DataFrame({
    "fragmento": [
        "El servicio es pésimo.",
        "La plataforma funciona bien.",
        "La atención no fue mala.",
    ],
    "con contexto": [
        'Una usuaria escribió: "El servicio es pésimo". La nota no aporta otros testimonios.',
        "La plataforma funciona bien para consultar archivos, pero no permite entregar las actividades.",
        "La atención no fue mala, aunque tampoco resolvieron mi consulta.",
    ],
})

resultados_fragmentos = analizar_textos(pd.DataFrame({"texto": pares_de_contexto["fragmento"]}))
resultados_con_contexto = analizar_textos(pd.DataFrame({"texto": pares_de_contexto["con contexto"]}))

pares_de_contexto["etiqueta del fragmento"] = resultados_fragmentos["predicción"]
pares_de_contexto["etiqueta con contexto"] = resultados_con_contexto["predicción"]
display(pares_de_contexto)

**En gráficos.** Cómo reparte el modelo su 100 % con la frase sola y con su contexto.

In [ ]:
dibujar_probabilidades(tabla=resultados_fragmentos, columna_texto="texto", título="Solo el fragmento")
dibujar_probabilidades(tabla=resultados_con_contexto, columna_texto="texto", título="El fragmento dentro de su contexto")

**Leé la tabla y los gráficos.**

- **Observación:** la cita del servicio "pésimo" se mantiene `NEG` con o sin contexto. "La plataforma funciona bien" pasa de `POS` a `NEU` cuando se agrega que no permite entregar actividades. "La atención no fue mala" pasa de `NEU` a `NEG` con el agregado "tampoco resolvieron mi consulta".
- **Interpretación posible:** agregar contexto no solo da más información: también cambia **qué estamos clasificando**.
- **Límite:** el modelo no distingue entre "lo que dice la cita" y "la posición de quien la presenta".

No es lo mismo preguntar:

- ¿Qué valoración expresa la cita?
- ¿Qué valoración expresa el texto completo?
- ¿Qué posición adopta quien presenta esa cita?

### ✎ Para pensar

Elegí un par y formulá una pregunta precisa. Explicá:

- Qué unidad analizaste.
- Qué resultado observaste.
- Qué interpretación proponés.
- Qué no podés concluir con esa clasificación.

Respuesta:

## 8. Guardar resultados y registro del entorno

**Operación.** Guardamos las tablas de resultados en archivos CSV y escribimos un archivo JSON con las versiones y el modelo usados. Así, quien quiera repetir el análisis sabe con qué condiciones se obtuvo.

**Advertencia:** esta celda crea la carpeta `salidas/` junto al cuaderno y sobrescribe los archivos si ya existen.

In [ ]:
from pathlib import Path
import json

# Path arma rutas de carpetas y archivos; mkdir crea la carpeta si no existe.
carpeta = Path("salidas")
carpeta.mkdir(exist_ok=True)

resultados.to_csv(carpeta / "pysentimiento_prueba.csv", index=False, encoding="utf-8")
resultados_desafíos.to_csv(carpeta / "pysentimiento_desafios.csv", index=False, encoding="utf-8")

**Operación.** Registramos con qué versiones y con qué modelo se obtuvieron estos resultados.

In [ ]:
entorno = {
    "python": sys.version.split()[0],
    "pandas": importlib.metadata.version("pandas"),
    "torch": importlib.metadata.version("torch"),
    "transformers": importlib.metadata.version("transformers"),
    "pysentimiento": importlib.metadata.version("pysentimiento"),
    "modelo_informado": str(getattr(configuración, "_name_or_path", "No informado")),
    "corpus": "Ejemplos construidos para una actividad didáctica",
}

# with abre el archivo y lo cierra solo al terminar; json.dump escribe el diccionario como JSON.
with open(carpeta / "entorno_pysentimiento.json", "w", encoding="utf-8") as archivo:
    json.dump(entorno, archivo, ensure_ascii=False, indent=2)

print("Resultados y registro del entorno guardados en", carpeta)

## Cierre

### Qué aprendimos

| Idea | Evidencia en este cuaderno |
|---|---|
| Usar un modelo no es lo mismo que entenderlo. | Solo pudimos observar sus respuestas; no sabemos por qué decide así. |
| Una probabilidad alta no es una garantía. | D05 y D07 con 92 % y 95 %, y ambos admiten lecturas humanas distintas. |
| Preprocesar también es decidir. | La limpieza agresiva igualó dos textos de sentido opuesto. |
| El contexto cambia lo que se clasifica. | La misma frase cambió de etiqueta al agregarle una oración. |

### Limitaciones de este análisis

- Corpus pequeño y construido; los resultados no se pueden generalizar.
- No sabemos con qué datos se entrenó el modelo más allá de lo que diga su ficha.
- Solo miramos etiquetas y probabilidades, no el funcionamiento interno del modelo.

### Conclusión: completá con tus palabras

- En este cuaderno hicimos inferencia, no entrenamiento, porque...
- El modelo utilizado fue...
- Un resultado compatible con mi lectura fue...
- Un desacuerdo que pude explicar fue...
- Una probabilidad alta no garantiza...
- Para aplicar el modelo a otro corpus necesitaríamos...

### Comparación de los tres enfoques

Completá usando resultados de tus ejecuciones de los tres cuadernos.

| Dimensión | Léxico y reglas (01) | Clasificador TF-IDF (02) | Modelo preentrenado (03) |
|---|---|---|---|
| De dónde proviene el criterio | | | |
| Datos utilizados para construirlo | | | |
| Decisiones que podemos inspeccionar | | | |
| Ejemplo resuelto satisfactoriamente | | | |
| Error o ambigüedad | | | |
| Dependencias y recursos necesarios | | | |
| Uso que recomendaríamos | | | |

### Afirmación final

Escribí un párrafo que incluya:

1. Una comparación delimitada a estos ejemplos.
2. Un fragmento que funcione como evidencia.
3. Un contraejemplo o limitación.
4. Una explicación de por qué los resultados no equivalen a medir "cómo se sienten los argentinos".

**Párrafo final:**